# Controlled 2x6 Factorial Benchmark Pipeline
## A Data-Centric Benchmark for Multi-Class Power-Line Defect Detection
**Authors:** Phạm Hoàng Minh Châu, Đỗ Trung Quân, Nguyễn Nhật Thông  
**Mentor:** Đặng Ngọc Minh Đức  

This notebook orchestrates the end-to-end benchmark workflow:
1. **Environment Setup & GPU Verification (RTX 4060)**
2. **Dataset Configuration:** Raw Combined (V6.2E) vs Curated Combined (8-rule curation)
3. **Resumable Training Runner:** 12 experimental variants (2 datasets × 6 models)
4. **Evaluation:** Four RT-DETR checkpoints on curated val_clean, then all 12 best checkpoints on the extracted test set
5. **Factorial Analysis & Reporting:** Data Curation Gain vs Model Scaling Gain

In [1]:
# Cell 1: Environment Setup & Module Imports
import sys, os, time
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local src is in path
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "benchmakring_transmission_line_defect_model").exists():
    CURRENT_DIR = CURRENT_DIR / "benchmakring_transmission_line_defect_model"
    os.chdir(CURRENT_DIR)
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))

from src.config import get_benchmark_config, get_class_names
from src.dataset import setup_dataset_yaml, verify_dataset_structure
from src.trainer import BenchmarkTrainer
from src.evaluator import BenchmarkEvaluator
from src.report import BenchmarkReporter

print("=== Environment Verification ===")
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("Classes (6-class taxonomy):", get_class_names())

=== Environment Verification ===
PyTorch Version: 2.14.1+cu126
CUDA Available : True
GPU Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM (GB): 8.0
Classes (6-class taxonomy): {0: 'insulator', 1: 'insulator_broken', 2: 'insulator_flashover', 3: 'bird_nest', 4: 'foreign_object', 5: 'broken_strand'}


In [9]:
# Cell 2: Dataset Verification (Raw Combined vs Curated Combined)
cfg = get_benchmark_config()
raw_dir = CURRENT_DIR / "data" / "raw_combined"
curated_dir = CURRENT_DIR / "data" / "curated_combined"

raw_stats = verify_dataset_structure(raw_dir, "val")
curated_stats = verify_dataset_structure(curated_dir, "val_clean")

stats_df = pd.DataFrame([
    {"Dataset": "Raw Combined (V6.2E uncurated)", "Train Images": raw_stats["train"]["images"], "Train Labels": raw_stats["train"]["labels"], "Val/Val_clean": raw_stats["val"]["images"], "YAML": cfg["datasets"]["raw"]["yaml_path"]},
    {"Dataset": "Curated Combined (8-rule curation)", "Train Images": curated_stats["train"]["images"], "Train Labels": curated_stats["train"]["labels"], "Val/Val_clean": curated_stats["val"]["images"], "YAML": cfg["datasets"]["curated"]["yaml_path"]}
])
print("=== Dataset Verification Status ===")
display(stats_df)

=== Dataset Verification Status ===


,Dataset,Train Images,Train Labels,Val/Val_clean,YAML
0,Raw Combined (V6.2E uncurated),18334,18334,2272,configs/dataset_raw.yaml
1,Curated Combined (8-rule curation),17903,17903,2271,configs/dataset_curated.yaml


In [ ]:
# Cell 3: Training Management (Fully Resumable 2x6 Factorial)
trainer = BenchmarkTrainer()

# Display current status of all 12 runs
status_table = pd.DataFrame(list(trainer.get_run_status().items()), columns=["Run Key", "Status"])
display(status_table)

# To train a single variant:
# trainer.train_single_run(dataset_type="raw", model_key="yolo11n")

# To launch the full benchmark suite (automatically resumes if stopped!)    :
trainer.run_all_benchmarks(datasets=["raw", "curated"], models=["yolo11n", "yolo11m", "yolo26n", "yolo26m"])

In [2]:
# Cell 4: Evaluate best checkpoints on curated val_clean and the extracted test set
import yaml

val_clean_source_yaml = CURRENT_DIR / "data" / "curated_combined" / "data_val_clean.yaml"
val_clean_results_dir = CURRENT_DIR / "results_val_clean"
val_clean_results_dir.mkdir(parents=True, exist_ok=True)
val_clean_yaml = val_clean_results_dir / "data_val_clean.yaml"
test_yaml = CURRENT_DIR.parent / "data" / "working" / "test" / "dataset.yaml"
for dataset_yaml in (val_clean_source_yaml, test_yaml):
    if not dataset_yaml.is_file():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

with val_clean_source_yaml.open("r", encoding="utf-8") as f:
    val_clean_data = yaml.safe_load(f)
val_clean_data["path"] = str((CURRENT_DIR / "data" / "curated_combined").resolve())
with val_clean_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(val_clean_data, f, sort_keys=False)

with test_yaml.open("r", encoding="utf-8") as f:
    test_data = yaml.safe_load(f)
if not test_data.get("train"):
    test_data["train"] = test_data["test"]
    with test_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(test_data, f, sort_keys=False)

all_run_names = [
    f"{dataset}_{model}"
    for dataset in ("raw", "curated")
    for model in ("yolo11n", "yolo11m", "yolo26n", "yolo26m", "rtdetr50", "rtdetr100")
]
checkpoint_paths = {
    run_key: CURRENT_DIR / "runs" / run_key / "weights" / "best.pt"
    for run_key in all_run_names
}
missing_checkpoints = [str(path) for path in checkpoint_paths.values() if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Missing best.pt checkpoints:\n" + "\n".join(missing_checkpoints))

def create_evaluator(results_dir):
    results_dir.mkdir(parents=True, exist_ok=True)
    instance = BenchmarkEvaluator()
    instance.results_dir = results_dir
    instance.eval_file = results_dir / "benchmark_test_results.json"
    instance.eval_data = instance._load_results()
    return instance

# val_clean_run_names = (
#     "raw_rtdetr50",
#     "raw_rtdetr100",
#     "curated_rtdetr50",
#     "curated_rtdetr100",
# )
# val_clean_runs = {
#     run_key: {"best_weights": str(checkpoint_paths[run_key])}
#     for run_key in val_clean_run_names
# }
# val_clean_evaluator = create_evaluator(val_clean_results_dir)
# print(f"Evaluating four RT-DETR best.pt checkpoints on curated val_clean: {val_clean_yaml}")
# df_val_clean = val_clean_evaluator.evaluate_all(
#     val_clean_runs,
#     val_clean_yaml,
#     split_by_dataset={"raw": "val", "curated": "val"},
#     checkpoint_field="best_weights",
# )
# display(df_val_clean)

test_runs = {
    run_key: {"best_weights": str(checkpoint_path)}
    for run_key, checkpoint_path in checkpoint_paths.items()
}
test_evaluator = create_evaluator(CURRENT_DIR / "results")
print(f"Evaluating all 12 best.pt checkpoints on the extracted test set: {test_yaml}")
df_test = test_evaluator.evaluate_all(
    test_runs,
    test_yaml,
    split_by_dataset={"raw": "test", "curated": "test"},
    checkpoint_field="best_weights",
)
display(df_test)


Evaluating all 12 best.pt checkpoints on the extracted test set: d:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\test\dataset.yaml

--- [EVALUATE] Evaluating 'raw_yolo11n' on split 'test' ---
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 131.037.9 MB/s, size: 42.8 KB)
val: Scanning D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\test\labels\test.cache... 1931 images, 145 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1931/1931  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 121/121 3.3it/s 36.2s0.1s5
                   all       1931       2562      0.814      0.665      0.736      0.403
Speed: 1.2ms preprocess, 2.0ms inference, 0.0ms loss, 1.4ms postpr

,Run,Dataset,Model,Evaluation Split,Precision,Recall,F1-Score,mAP@0.5,mAP@0.5:0.95,MEAN_5,Latency (ms),FPS,GFLOPS,Params (M),VRAM (MB)
0,raw_yolo11n,RAW,yolo11n,test,0.8142,0.6650,0.7321,0.7356,0.4029,0.6956,16.87,59.3,6.50,2.59,75.9
1,raw_yolo11m,RAW,yolo11m,test,0.8018,0.7344,0.7666,0.7803,0.4287,0.7474,13.95,71.7,68.32,20.06,267.2
2,raw_yolo26n,RAW,yolo26n,test,0.8177,0.7264,0.7694,0.7831,0.4281,0.7520,20.35,49.1,5.90,2.51,228.9
3,raw_yolo26m,RAW,yolo26m,test,0.8593,0.7539,0.8031,0.8119,0.4497,0.7817,21.94,45.6,75.00,21.78,271.4
4,raw_rtdetr50,RAW,rtdetr50,test,0.8344,0.7489,0.7891,0.7536,0.4113,0.7254,35.72,28.0,137.72,42.74,437.1
5,raw_rtdetr100,RAW,rtdetr100,test,0.9019,0.4581,0.5800,0.7551,0.4005,0.7344,49.74,20.1,260.61,76.40,569.7
6,curated_yolo11n,CURATED,yolo11n,test,0.8089,0.6705,0.7333,0.7436,0.4114,0.7079,11.24,89.0,6.50,2.59,77.2
7,curated_yolo11m,CURATED,yolo11m,test,0.8357,0.7168,0.7717,0.7734,0.4230,0.7377,13.56,73.7,68.32,20.06,289.7
8,curated_yolo26n,CURATED,yolo26n,test,0.8246,0.7308,0.7749,0.7775,0.4264,0.7455,13.27,75.4,5.90,2.51,75.4
9,curated_yolo26m,CURATED,yolo26m,test,0.8688,0.7399,0.7992,0.8172,0.4484,0.7892,14.76,67.8,75.00,21.78,289.7


In [ ]:
# Cell 5: 2x6 Factorial Matrix, Visualization & Report Generation
reporter = BenchmarkReporter()
report_md = reporter.generate_report()

# Load matrix for visual plotting
matrix_csv = CURRENT_DIR / "results" / "factorial_benchmark_matrix.csv"
if matrix_csv.exists():
    df_mat = pd.read_csv(matrix_csv)
    display(df_mat)
    
    # Bar chart: Raw vs Curated mAP@0.5
    if "Raw mAP50" in df_mat.columns and "Curated mAP50" in df_mat.columns:
        plt.figure(figsize=(10, 5))
        x = range(len(df_mat))
        # Convert numeric if available
        try:
            raw_vals = pd.to_numeric(df_mat["Raw mAP50"], errors='coerce')
            cur_vals = pd.to_numeric(df_mat["Curated mAP50"], errors='coerce')
            plt.bar([i - 0.2 for i in x], raw_vals, width=0.4, label="Raw Combined", color="salmon")
            plt.bar([i + 0.2 for i in x], cur_vals, width=0.4, label="Curated Combined", color="teal")
            plt.xticks(x, df_mat["Architecture"])
            plt.ylabel("mAP @ 0.5")
            plt.title("Controlled 2x6 Factorial: Raw vs Curated Data Performance")
            plt.legend()
            plt.grid(axis="y", linestyle="--", alpha=0.7)
            plt.show()
        except Exception as e:
            print("Plotting error:", e)

print("Report saved in: results/factorial_benchmark_report.md")